# 5.4 多流并发死锁检测功能

## 1. 功能简介

多流并发场景下存在一种隐蔽的死锁风险：当多个流上的通信类算子（如 all_gather、MoE 通信算子等）同时进入就绪状态，且它们请求的 AIV（Vector）核心数量总和超过硬件总可用数时，NPU 调度器将无法同时满足所有任务的资源分配需求——每个任务都在等待其他任务释放核心资源，但没有任何任务能够获得足够的资源开始运行，从而形成死锁。

本功能在**编译期**自动分析整图中各流上通信类算子的核心资源占用，提前发现此类死锁隐患，无需真正运行到死锁发生。

这类问题可以通过第四章介绍的多流表达功能（4.3）和限核功能（4.4）来规避：合理分配各流的核数上限，避免通信算子核数需求总和超过硬件总量。


## 2. 使用方法

通过 `npugraph_ex` 的 options 配置开启死锁检测，参数说明如下：

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr>
    <th align="left">参数名</th>
    <th align="left">参数说明</th>
  </tr>
  <tr>
    <td align="left">deadlock_check</td>
    <td align="left">是否开启死锁检测功能，bool 类型。False（默认值）：不开启死锁检测；True：开启死锁检测。</td>
  </tr>
</table>
<br clear="all">

下面的示例演示在多流表达模型上开启死锁检测，主要步骤如下：

1. **构造多流模型**：在 forward 中创建从流（Stream）和同步事件（Event），将矩阵乘法分发到从流执行，与默认流上的加法形成并行。
2. **正确同步**：通过 `record` 和 `wait` 控制流间时序，保证数据依赖正确。
3. **开启死锁检测**：编译时在 options 中设置 `deadlock_check` 为 True。
4. **执行并观察**：正常运行无告警输出；若检出死锁风险，将打印 CONFLICT 日志并生成检测产物。

>运行下方代码单元格，体验多流模型的死锁检测开启方式。

In [ ]:
import torch
import torch_npu

assert torch.npu.is_available(), "请在昇腾 NPU 环境运行本示例"
torch.manual_seed(0)

class MultiStreamDemo(torch.nn.Module):
    def forward(self, in1, in2):
        # 创建从流和同步事件
        stream1 = torch.npu.Stream()
        event1 = torch.npu.Event()
        event2 = torch.npu.Event()

        # 默认流上执行加法
        add_result = torch.add(in1, in2)
        # 记录 event1，供从流同步
        event1.record()
        with torch.npu.stream(stream1):
            # 从流等待默认流上的 add 完成
            event1.wait(stream1)
            # 从流上执行矩阵乘
            mm_result = torch.mm(in1, in2)
            # 记录 event2，供默认流同步
            event2.record()
        # 默认流等待从流上的 mm 完成
        torch.npu.current_stream().wait_event(event2)
        return add_result, mm_result

model = MultiStreamDemo().npu().eval()
# 编译时开启死锁检测
compiled = torch.compile(
    model,
    backend="npugraph_ex",
    options={"deadlock_check": True},
    dynamic=False,
)

in1 = torch.randn(1000, 1000, dtype=torch.float16).npu()
in2 = torch.randn(1000, 1000, dtype=torch.float16).npu()
with torch.no_grad():
    add_result, mm_result = compiled(in1, in2)
torch.npu.synchronize()
print("add_result shape:", tuple(add_result.shape))
print("mm_result shape:", tuple(mm_result.shape))
print("正常多流模型未检出死锁风险；若检出，将打印 CONFLICT 日志并生成检测 json。")

## 3. 产物说明

若检出死锁风险，会在脚本执行同级目录下生成如下产物（目录名中的时间戳和进程号与实际运行相关）：

```txt
torch_compile_debug
└── run_<时间戳>-pid_<进程号>
    └── npugraph_ex
        └── model__xx
            ├── graph_1_id_<id>_rank_<rank>_pid_<pid>_ts_<ts>.json # 原始 dump json
            ├── graph_1_id_<id>_rank_<rank>_pid_<pid>_ts_<ts>_filtered.json  # 过滤掉计算算子后的 json
            └── graph_1_id_<id>_rank_<rank>_pid_<pid>_ts_<ts>_filtered_deadlock_check.json # 死锁检测结果 json
```

同时会有打屏日志提示：

```bash
[RESULT] Deadlock risk detected! 56 conflicting pair(s):
  CONFLICT: [stream316 TaskId=4] aiv_all_gather_bfloat16_t (32 AIVEC)  <->  [stream317 TaskId=5] MoeDistributeDispatchV2_... (24 AIVEC)  sum=56 > 48
```

### 日志解读

以上面这条 CONFLICT 为例，各字段含义如下：

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr>
    <th align="left">字段</th>
    <th align="left">含义</th>
  </tr>
  <tr>
    <td align="left">stream316 / stream317</td>
    <td align="left">冲突算子所在的流编号</td>
  </tr>
  <tr>
    <td align="left">TaskId=4 / TaskId=5</td>
    <td align="left">任务在流内的编号</td>
  </tr>
  <tr>
    <td align="left">aiv_all_gather_bfloat16_t / MoeDistributeDispatchV2</td>
    <td align="left">算子名称，均为通信类算子</td>
  </tr>
  <tr>
    <td align="left">32 AIVEC / 24 AIVEC</td>
    <td align="left">各算子占用的 Vector 核数量</td>
  </tr>
  <tr>
    <td align="left">sum=56 > 48</td>
    <td align="left">两算子核数总和 56 超过设备总核数 48，存在死锁风险</td>
  </tr>
</table>
<br clear="all">

设备总核数可通过"CANN 软件安装目录/`<arch>`-linux/data/platform_config/`<soc_version>`.ini"查看，其中 `vector_core_cnt=48` 即本例中的 48 个 Vector Core。

死锁的 task 统计保存在 `*_filtered_deadlock_check.json` 文件中，可将该产物连同版本信息一起归档，用于问题上报和修复验证。

> 本节示例模型未包含多流通信类算子，因此不会触发告警。真实风险常见于多卡分布式推理（如 MoE 模型中 all_gather 与 Dispatch/Combine 通信算子被划分到不同流并行执行）的场景。修复手段可参考第四章限核功能（4.4），为各流设置合理的核数上限。

## 4. 课后练习

### 一、单选题

1. 多流并发场景下，本功能所检测的死锁风险是如何形成的？
- A. 显存不足导致算子内存分配失败
- B. 输入 Shape 不一致导致图编译失败
- C. Event 未正确 record 导致流同步失败
- D. 多个流上通信类算子请求的 AIV 核心总数超过硬件总可用数

2. 本功能在哪个阶段完成死锁风险分析？
- A. 编译期
- B. 运行期
- C. 模型加载期
- D. 数据预处理期

3. 开启死锁检测功能的正确配置方式是？
- A. 终端执行 export DEADLOCK_CHECK=1
- B. torch.compile(..., fullgraph=True)
- C. 编译时在 options 中设置 deadlock_check 为 True
- D. 在模型 forward 中调用 deadlock_check() 接口

4. CONFLICT 日志中 `sum=56 > 48` 的含义是？
- A. 两个算子的 TaskId 之和为 56
- B. 冲突算子占用的 Vector 核数总和 56 超过设备总核数 48
- C. 两个流的编号相加超过上限
- D. 显存占用 56GB 超过总显存 48GB

5. 设备总核数（如 48 个 Vector Core）可通过哪个文件查看？
- A. /root/ascend/log 下的日志文件
- B. torch_compile_debug/run_*/debug.log
- C. 检测产物 json 文件的头部注释
- D. CANN 安装目录下 platform_config/<soc_version>.ini

6. 检出死锁风险后，保存死锁 task 统计信息的产物文件是？
- A. dynamo_out_graph.txt
- B. 000_aot_forward_graph.txt
- C. *_filtered_deadlock_check.json
- D. output_code.py

### 二、多选题

7. 关于多流并发死锁风险的成因与常见场景，正确的说法有哪些？
- A. 涉及的算子通常是通信类算子，如 all_gather、MoE 通信算子
- B. 真实风险常见于多卡分布式推理，如 MoE 模型通信算子被划分到不同流并行执行
- C. 单流顺序执行的模型同样会触发本功能告警
- D. 本节示例模型必然触发 CONFLICT 告警

8. 检出死锁风险后，关于产物与后续处理，正确的说法有哪些？
- A. 产物生成在 torch_compile_debug/run_*/npugraph_ex/model__xx/ 目录下
- B. 死锁的 task 统计保存在 *_filtered_deadlock_check.json 文件中
- C. 可将该产物连同版本信息一起归档，用于问题上报和修复验证
- D. 产物会自动上报给技术支持，无需人工处理

9. 关于本功能的行为特点，正确的说法有哪些？
- A. 必须真正运行到死锁发生才能检出风险
- B. 正常运行的多流模型若无风险，不会有告警输出
- C. 检出风险时会打印 CONFLICT 日志并生成检测产物
- D. 本功能不会自动修改各流核数配置，需结合限核功能进行修复

10. 关于多流模型中 Stream 和 Event 的用法，正确的说法有哪些？
- A. 通过 Event 的 record 和 wait 控制流间时序
- B. 从流执行算子前，应通过事件等待其依赖的上游结果完成
- C. 默认流可通过 wait_event 等待从流上记录的事件
- D. Event 无需 record 即可直接被其他流 wait

**运行以下代码单元查看参考答案与解析。**

In [ ]:
from pathlib import Path
answer = Path('answer/05.04_answer.txt')
print(answer.read_text(encoding='utf-8'))